# Walkthrough: from a recording to the results

This notebook goes through the analysis step by step: loading a recording, filtering the ECG and the PPG,
finding the heartbeats and breaths, checking the signal quality, following a whole isoflurane session and
comparing the values with published data. Every step uses the `mpcb` package of this folder.

To run it: install the packages once (`py -3.13 -m pip install -r requirements.txt`), open this notebook in
VS Code or Jupyter, and run the cells from top to bottom. The recordings are in `data/` as zip files and are
unzipped the first time they are used.

Every function has a description of what it does and how; show it with `help()`, for example
`help(filters.ecg_highpass)`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mpcb import SESSIONS, Recording, Session, filters, plotting, stats

plotting.figure_style()
%matplotlib inline

## 1. Load a recording

The GUI saves one CSV file per recording. Its first line holds the settings (sampling rates, ECG gain, PPG
light range); then follows one row per sample: ECG rows (512 per second), PPG rows (200 or 50 per second)
and the notes typed during the recording.

`Recording` reads such a file. Without a file name it reads the default recording: 22 Oct, 14:54:30,
Mouse 1 (30 g), back at 2 % isoflurane after 4 %. Any other file of `data/` can be given instead.

In [ ]:
rec = Recording()
rec

In [ ]:
rec.settings

In [ ]:
rec.notes

The recording now holds the signals as arrays: `rec.ecg` (µV), `rec.t_ecg` (s), `rec.ir_nA` and `rec.red_nA`
(PPG light in nA), `rec.t_ppg`, `rec.temp_c` and `rec.t_temp`. The PPG times are on the ECG's clock (see
`help(Recording.__init__)`).

## 2. The raw signals

Two seconds of the ECG as recorded, 30 s of the infrared light and the skin temperature of the whole
recording. The raw ECG still holds slow drift and the 50 Hz mains.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7, 6), layout="constrained")
part = (rec.t_ecg >= 300) & (rec.t_ecg < 302)
axes[0].plot(rec.t_ecg[part], rec.ecg[part])
axes[0].set(xlabel="Time (s)", ylabel="ECG (µV)", title="ECG as recorded")
part = (rec.t_ppg >= 300) & (rec.t_ppg < 330)
axes[1].plot(rec.t_ppg[part], rec.ir_nA[part], linewidth=0.7)
axes[1].set(xlabel="Time (s)", ylabel="IR light (nA)", title="PPG as recorded")
axes[2].plot(rec.t_temp / 60, rec.temp_c)
axes[2].set(xlabel="Time (min)", ylabel="°C", title="Skin temperature")
plt.show()

## 3. Preprocessing: filtering the ECG

`rec.ecg_filtered()` applies two filters (both run forwards and backwards, so no wave is shifted in time):

- a 3 Hz high-pass, which removes the slow drift and most of the breathing;
- notches at 50 and 150 Hz, which remove the mains and its 3rd harmonic.

The ECG chip has already removed everything above about 150 Hz before the data were stored, so no low-pass
is added. The spectra show what each step removes.

In [ ]:
print("high-pass:", filters.ECG_HP_HZ, "Hz; notches:", filters.NOTCH_HZ, "Hz")
ecg = rec.ecg_filtered()
fig, axes = plt.subplots(1, 2, figsize=(9, 3), layout="constrained")
part = (rec.t_ecg >= 300) & (rec.t_ecg < 301)
axes[0].plot(rec.t_ecg[part], rec.ecg[part] - np.mean(rec.ecg[part]), color="0.7", label="raw")
axes[0].plot(rec.t_ecg[part], ecg[part], label="filtered")
axes[0].set(xlabel="Time (s)", ylabel="ECG (µV)")
axes[0].legend()
plt.sca(axes[1])
plotting.step3_before_after(rec.ecg, ecg, rec.fs_ecg)
plt.show()

## 4. R waves and heart rate

`rec.r_waves()` finds the R wave of every beat (see `r_waves` in `mpcb/analysis.py`); it prints in how many 30 s
pieces the heartbeat repeats clearly. `rec.heart_rate()` gives the heart rate every 10 s from the time
between R waves. `rec.averaged_beat()` lines all beats up on their R wave and takes the median, so the small
P wave and the J wave become visible.

In [ ]:
r = rec.r_waves()
t_hr, hr = rec.heart_rate(r=r)
t_ms, beat = rec.averaged_beat(r=r)

fig, axes = plt.subplots(1, 3, figsize=(11, 3), layout="constrained")
part = (rec.t_ecg >= 300) & (rec.t_ecg < 302)
axes[0].plot(rec.t_ecg[part], ecg[part])
marks = r[(rec.t_ecg[r] >= 300) & (rec.t_ecg[r] < 302)]
axes[0].plot(rec.t_ecg[marks], ecg[marks], "v")
axes[0].set(xlabel="Time (s)", ylabel="ECG (µV)", title="R waves found")
axes[1].plot(t_hr / 60, hr)
axes[1].set(xlabel="Time (min)", ylabel="beats per minute", title="Heart rate every 10 s")
axes[2].plot(t_ms, beat)
axes[2].set(xlabel="Time from R (ms)", ylabel="ECG (µV)", title="Averaged beat")
plt.show()
print("median heart rate: %.0f beats per minute" % np.nanmedian(hr))

## 5. The PPG: pulse and breathing

The infrared light is low-passed at 32 Hz and split into two bands: the heart band (4-11 Hz) and the
breathing band (0.15-3.5 Hz).

- `rec.averaged_pulse()` averages the heart band on the R waves of the ECG. Because the ECG is a separate
  sensor, a pulse that appears here comes from the heartbeat.
- `rec.breathing_rate()` gives one breathing rate for a stretch of time, from how alike the breathing band
  is to itself shifted by one breath.
- `rec.breaths()` finds every single breath, which also works when breathing is irregular.

In [ ]:
breath = rec.breathing_band()
t_pulse, pulse = rec.averaged_pulse(r=r)
t_breaths, rates = rec.breaths()

fig, axes = plt.subplots(1, 3, figsize=(11, 3), layout="constrained")
part = (rec.t_ppg >= 300) & (rec.t_ppg < 330)
axes[0].plot(rec.t_ppg[part], breath[part])
axes[0].set(xlabel="Time (s)", ylabel="nA", title="Breathing band")
axes[1].plot(t_pulse, pulse)
axes[1].set(xlabel="Time from R (ms)", ylabel="nA", title="Pulse averaged on the R waves")
axes[2].plot(t_breaths / 60, rates, ".", markersize=2)
axes[2].set(xlabel="Time (min)", ylabel="breaths per minute", title="Single breaths")
plt.show()
print("breathing rate, 300 to 330 s: %.0f per minute" % rec.breathing_rate(300, 330))
print("pulse size: %.1f nA, %.2f %% of the light" % (np.ptp(pulse), 100 * np.ptp(pulse) / np.mean(rec.ir_nA)))

## 6. Signal quality

The checks of a recording are methods of `Recording`:

- `rec.lost_samples()`: samples lost in gaps, where a step between time stamps is longer than one sample;
- `rec.mains_hz()`: where the mains shows; samples lost without a gap speed the signal up and move it above 50 Hz;
- `rec.clipped()`: ECG samples at the end of the chip's range, in %;
- `rec.beat_quality()`: in each 30 s piece in which the heartbeat repeats clearly, the signal-to-noise ratio of the
  ECG and the PPG (what repeats with every beat against what does not, in dB; see `snr_db` in `mpcb/analysis.py`) and the
  heart-rate variability (SDNN, RMSSD).

`scripts/quality.py` puts the same values of every recording in one table.

In [ ]:
print("ECG samples lost: %d; mains at %.2f Hz; ECG clipped: %.2f %%" % (rec.lost_samples(), rec.mains_hz(), rec.clipped()))
pieces, snr_ecg, snr_ppg, sdnn, rmssd = rec.beat_quality()
print("median of %d pieces: ECG SNR %.1f dB, PPG SNR %.1f dB, SDNN %.2f ms, RMSSD %.2f ms"
      % (len(snr_ecg), np.median(snr_ecg), np.median(snr_ppg), np.nanmedian(sdnn), np.nanmedian(rmssd)))

## 7. A whole isoflurane session

A `Session` puts all recordings of one session on one clock and reads the isoflurane dose from the notes
("iso to 2" is 2 %). `s.vitals()` gives heart rate, breathing, perfusion index, skin temperature, SNR and
heart-rate variability every 10 s (this takes about half a minute). The sessions are:

In [ ]:
list(SESSIONS)

In [ ]:
s = Session("26 Oct, 30 g")
v = s.vitals()
s.doses()

In [ ]:
t_breaths, rates = s.breaths()
fig, axes = plt.subplots(4, 1, figsize=(7, 8), sharex=True, layout="constrained")
when = [c[0] / 60 for c in v["changes"]] + [v["end"] / 60]
axes[0].stairs([c[1] for c in v["changes"]], when, baseline=None, zorder=3)
axes[0].set(ylabel="Isoflurane (%)")
axes[1].plot(v["t"] / 60, v["hr"])
axes[1].set(ylabel="Heart rate (per min)")
axes[2].plot(t_breaths / 60, rates, ".", color="0.6", markersize=2)
axes[2].plot(v["t"] / 60, v["breathing"])
axes[2].set(ylabel="Breathing (per min)", ylim=(0, 260))
axes[3].plot(v["t"] / 60, v["temp"])
axes[3].set(ylabel="Skin temp. (°C)", xlabel="Time (min)")
for ax in axes:
    ax.tick_params(labelbottom=True)
plt.show()

## 8. Values per dose and comparison with published values

`stats.per_dose()` takes the mean of the last 2 min at 1, 2 and 4 % in each of the three runs that went
through these doses (Mouse 1 on 22 and 26 Oct, Mouse 2 on 22 Oct). Their mean ± SD can be compared with a
published mean ± SD by a t-test (`stats.t_test`, the same test as the MedCalc comparison-of-means
calculator). With three runs from two mice the test is only a first indication.

In [ ]:
hr = stats.per_dose("hr")
for dose, values in hr.items():
    mean, sd, n = stats.mean_sd(values)
    print("%g %%: %.0f ± %.1f beats per minute (n = %d)" % (dose, mean, sd, n))

# published: Picard et al. (2014), light anaesthesia (0.5 % isoflurane), 522.5 ± 16.7, n = 12
mean, sd, n = stats.mean_sd(hr[1])
t, p = stats.t_test(mean, sd, n, 522.5, 16.72, 12)
print("1 %% against Picard et al. (2014): t = %.2f, p = %.2f" % (t, p))

## 9. The scripts that make the figures and tables

| Script | What it makes |
|---|---|
| `scripts/ecg_filter_steps.py` | the ECG filtering steps with their spectra |
| `scripts/ppg_vitals.py` | the PPG filtering steps and the vitals every 30 s |
| `scripts/notch_differences.py` | what each mains notch removes |
| `scripts/quality.py` | the signal quality of every recording (`quality.csv`) |
| `scripts/hr_agreement.py` | PPG against ECG heart rate (Bland-Altman) |
| `scripts/checks.py` | the methods tried on made-up signals with a known answer |
| `isoflurane/timeline.py`, `doses.py`, `table.py` | one session over time, per dose, and the tables |
| `report/figures.py` | the graphs of the results report |

Run them from the main folder, e.g. `py -3.13 scripts/ecg_filter_steps.py`. Their figures and tables are saved in `outputs/`.